# TakeMeter — Fine-Tuning Starter Notebook
## AI201 · Project 3

This notebook walks you through fine-tuning a text classifier on your annotated dataset and comparing it to a zero-shot baseline.

**What this notebook does for you (infrastructure):**
- Tokenizes your dataset and prepares it for training
- Runs the fine-tuning pipeline with DistilBERT
- Computes evaluation metrics and generates a confusion matrix
- Runs the Groq baseline and compares both models

**What you do (the actual work):**
- Collect and annotate your 200+ examples (done before opening this notebook)
- Define your label map and upload your CSV
- Write your Groq classification prompt using your label definitions
- Analyze the output and write your evaluation report

**Before you start:** Make sure you are using a T4 GPU runtime. Go to Runtime → Change runtime type → T4 GPU, then click Save.

In [ ]:
# Install any dependencies not pre-installed on Colab
!pip install -q groq python-dotenv
print("✅ Dependencies ready")

: 

In [ ]:
import pandas as pd
import numpy as np
import json
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

print("✅ Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Section 1: Load Your Dataset

Upload your labeled CSV and define your label map. Your CSV must have at least two columns: `text` (the post/comment) and `label` (your string label).

In [ ]:
# ── Label map for r/television TakeMeter classifier ───────────────────────
# analysis   — evaluates writing/direction/themes using specific textual evidence
# reaction   — immediate emotional or evaluative response, little to no argument
# prediction — speculative forecast about future episodes/seasons
# ─────────────────────────────────────────────────────────────────────────

LABEL_MAP = {
    "analysis":   0,
    "reaction":   1,
    "prediction": 2,
}

ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)
print(f"Labels: {LABEL_MAP}")
print(f"Number of labels: {NUM_LABELS}")

In [ ]:
# Upload your CSV from your computer
from google.colab import files
print("Select your labeled dataset CSV file...")
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {CSV_PATH}")

In [ ]:
# Load and validate your dataset
df = pd.read_csv(CSV_PATH)

# If your CSV uses different column names, rename them here.
# Example: df = df.rename(columns={"post": "text", "category": "label"})

print(f"Columns: {df.columns.tolist()}")
print(f"Total examples: {len(df)}")
print()
print("Label distribution:")
print(df["label"].value_counts())

# Validate all labels are in LABEL_MAP
unknown = set(df["label"].unique()) - set(LABEL_MAP.keys())
if unknown:
    print(f"\n⚠️  Labels in CSV not found in LABEL_MAP: {unknown}")
    print("Update your LABEL_MAP above to include all labels.")
else:
    print("\n✅ All labels match your LABEL_MAP")

# Convert string labels to integers
df["label_id"] = df["label"].map(LABEL_MAP)
df = df.dropna(subset=["label_id"])
df["label_id"] = df["label_id"].astype(int)

## Section 2: Prepare Data for Training

Splits your dataset into train / validation / test sets and tokenizes the text.

In [ ]:
# Train / val / test split — 70% / 15% / 15%
# Stratified so each split has roughly the same label distribution.
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label_id"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label_id"]
)

print(f"Train: {len(train_df)} examples")
print(f"Validation: {len(val_df)} examples")
print(f"Test: {len(test_df)} examples")
print()
print("Train label distribution:")
print(train_df["label"].value_counts())
print()
print("Test label distribution:")
print(test_df["label"].value_counts())

# Reset indices (needed for clean HuggingFace Dataset conversion)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [ ]:
# Load tokenizer and tokenize all splits
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

def make_dataset(df_split):
    ds = Dataset.from_pandas(
        df_split[["text", "label_id"]].rename(columns={"label_id": "labels"})
    )
    return ds.map(tokenize, batched=True)

train_dataset = make_dataset(train_df)
val_dataset   = make_dataset(val_df)
test_dataset  = make_dataset(test_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("✅ Tokenization complete")
print(f"Sample keys: {list(train_dataset[0].keys())}")

## Section 3: Fine-Tune Your Model

Loads `distilbert-base-uncased` with a classification head and fine-tunes it on your training data. Training runs for 3 epochs and takes 5–15 minutes on a T4 GPU.

**Hyperparameter note:** The defaults below work well for datasets of 100–500 examples. If you change any values, note what you changed and why in your README.

In [ ]:
# Load DistilBERT with a classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
)
print(f"✅ Model loaded: {MODEL_NAME}")
print(f"Output labels: {NUM_LABELS}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────
# num_train_epochs  — passes through the training data; 3 is a good default
#                     for small datasets. Increase cautiously; more epochs
#                     risk overfitting on 200 examples.
# learning_rate     — 2e-5 is the standard starting point for fine-tuning
#                     BERT-family models. Lower → slower but more stable.
# per_device_train_batch_size — 16 fits T4 GPU comfortably.
#                     Reduce to 8 if you get out-of-memory errors.
# ─────────────────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./takemeter-model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning... (5–15 minutes on T4 GPU)")
trainer.train()
print("\n✅ Fine-tuning complete")

## Section 4: Evaluate Fine-Tuned Model on Test Set

Runs inference on your locked test set and generates metrics and a confusion matrix. These numbers go directly into your evaluation report.

In [ ]:
# Run inference on the test set
print("Running inference on test set...")
ft_output = trainer.predict(test_dataset)
ft_pred_ids = np.argmax(ft_output.predictions, axis=-1)
ft_true_ids = ft_output.label_ids

ft_probs = torch.nn.functional.softmax(
    torch.tensor(ft_output.predictions), dim=-1
).numpy()

# Overall accuracy
ft_accuracy = accuracy_score(ft_true_ids, ft_pred_ids)
print(f"\n🎯 Fine-tuned model accuracy: {ft_accuracy:.3f}")

# Per-class metrics
label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print("\nPer-class metrics (fine-tuned model):")
print(classification_report(ft_true_ids, ft_pred_ids, target_names=label_names, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(ft_true_ids, ft_pred_ids)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Fine-Tuned Model — Confusion Matrix (Test Set)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("✅ Saved: confusion_matrix.png  →  commit this to your repo and include in README")

In [ ]:
# Print wrong predictions for your error analysis
# Review these carefully — pick 3 to analyze in depth in your README.

wrong_idx = np.where(ft_pred_ids != ft_true_ids)[0]
print(f"Wrong predictions: {len(wrong_idx)} / {len(ft_true_ids)}\n")

for i, idx in enumerate(wrong_idx[:15]):
    text = test_df.iloc[idx]["text"]
    true_label = ID_TO_LABEL[ft_true_ids[idx]]
    pred_label = ID_TO_LABEL[ft_pred_ids[idx]]
    confidence = ft_probs[idx][ft_pred_ids[idx]]
    print(f"--- #{i+1} ---")
    print(f"Text:      {text[:200]}{'...' if len(text) > 200 else ''}")
    print(f"True:      {true_label}")
    print(f"Predicted: {pred_label}  (confidence: {confidence:.2f})")
    print()

## Section 5: Baseline Classifier (Groq)

Runs your zero-shot baseline using `llama-3.3-70b-versatile`. You need to write the classification prompt using your label definitions.

**Add your Groq API key** via Colab Secrets (🔑 icon in the left sidebar → name: `GROQ_API_KEY`) before running this section.

In [ ]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
print("✅ Groq client ready")

In [ ]:
SYSTEM_PROMPT = """
You are classifying Reddit posts from r/television and r/WidowsBay discussing the show Widow's Bay (Apple TV+).
Assign each post to exactly one of the following labels.

analysis: A post that evaluates the show's writing, direction, performances, or themes using
specific, verifiable references to the show's text — scenes, dialogue, characters, structure —
rather than just delivering a verdict.
Example: "The Church Bells effectively being dinner bells was maybe my favourite punchline of
the season. It recontextualizes every scene in the first episode where they ring."

reaction: An immediate emotional or evaluative response to a specific episode or moment, with
little to no supporting reasoning. The post is expressing a feeling or verdict rather than
making an argument.
Example: "I shrieked with laughter when Dale told everyone to run. Then was immediately silenced
with a gasp of horror when Ruth was shot. Man this episode was so fucking good."

prediction: A speculative claim about what will happen in a future episode or season, stated as
a forecast. The primary content is the prediction itself, not evaluation of something that
already aired.
Example: "Prediction for season 2: Tom Wyck Bechir Patricia and maybe Dale will be the new
sacrifice committee."

Edge case rule: if a post mixes prediction and analysis, ask whether removing the speculative
sentence would leave a complete evaluative post. If yes → analysis. If the speculation is
the point → prediction.

Respond with ONLY the label name — one of: analysis, reaction, prediction.
No punctuation, no explanation, no other text.
"""

print("Prompt length:", len(SYSTEM_PROMPT), "characters")

In [ ]:
def classify_with_groq(text):
    """Classify a single post. Returns a label string or None if unparseable."""
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Classify this post:\n\n{text}"},
            ],
            temperature=0,
            max_tokens=20,
        )
        raw = response.choices[0].message.content.strip().lower()
        # Match the model's output to a label. Check longest labels first so a
        # label that is a substring of another can't be matched by mistake.
        for label in sorted(LABEL_MAP, key=len, reverse=True):
            if raw == label or label in raw:
                return label
        return None  # model output didn't match any known label
    except Exception as e:
        print(f"API error: {e}")
        return None


# Run baseline on test set
print(f"Running baseline on {len(test_df)} examples...")
print("(May take a few minutes — 0.1s delay between requests to respect free-tier limits)\n")

baseline_preds = []
for i, (_, row) in enumerate(test_df.iterrows()):
    pred = classify_with_groq(row["text"])
    baseline_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(test_df)} complete...")
    time.sleep(0.1)

none_count = baseline_preds.count(None)
if none_count > 0:
    print(f"\n⚠️  {none_count} responses could not be parsed.")
    print("Review your prompt — the model may not be outputting clean label names.")

In [ ]:
# Baseline metrics (exclude unparseable responses)
valid = [(p, t) for p, t in zip(baseline_preds, test_df["label_id"])
         if p is not None]
bl_pred_ids = [LABEL_MAP[p] for p, _ in valid]
bl_true_ids = [t for _, t in valid]

bl_accuracy = accuracy_score(bl_true_ids, bl_pred_ids)
print(f"🎯 Baseline accuracy: {bl_accuracy:.3f}  "
      f"(evaluated on {len(valid)}/{len(test_df)} parseable responses)")
print()
label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print("Per-class metrics (baseline):")
print(classification_report(bl_true_ids, bl_pred_ids, target_names=label_names, zero_division=0))

## Section 6: Compare Results and Export

Side-by-side comparison of both models. Download the output files and commit them to your GitHub repo.

In [ ]:
# ── Cell 1: Side-by-side comparison ────────────────────────────────────────

print("=" * 60)
print("MODEL COMPARISON — Test Set Results")
print("=" * 60)
print(f"\n{'Metric':<20}{'Fine-Tuned':<15}{'Baseline (Groq)':<15}")
print("-" * 50)
print(f"{'Accuracy':<20}{ft_accuracy:<15.3f}{bl_accuracy:<15.3f}")
print(f"{'Improvement':<20}{ft_accuracy - bl_accuracy:+.3f}")

print("\nFine-tuned per-class report:")
ft_report = classification_report(
    ft_true_ids, ft_pred_ids, target_names=label_names,
    zero_division=0, output_dict=True
)
print(classification_report(ft_true_ids, ft_pred_ids, target_names=label_names, zero_division=0))

print("\nBaseline per-class report:")
bl_report = classification_report(
    bl_true_ids, bl_pred_ids, target_names=label_names,
    zero_division=0, output_dict=True
)
print(classification_report(bl_true_ids, bl_pred_ids, target_names=label_names, zero_division=0))

# Confusion matrix as plain text — paste directly into your README as a markdown table
cm_dict = {
    "true_labels": label_names,
    "predicted_labels": label_names,
    "matrix": cm.tolist(),
}
print("\nConfusion matrix (rows=true, cols=predicted):")
header = "          " + "".join(f"{l[:10]:>12}" for l in label_names)
print(header)
for i, row in enumerate(cm):
    print(f"{label_names[i][:10]:<10}" + "".join(f"{v:>12}" for v in row))

print("\n→ Copy the matrix above into your README as a markdown table.")

In [ ]:
# ── Cell 2: Export evaluation_results.json ─────────────────────────────────

# Sample classifications for the README's "Sample Classifications" subsection
# (3 correct + 2 incorrect from the fine-tuned model)
correct_idx = list(np.where(ft_pred_ids == ft_true_ids)[0][:3])
sample_indices = correct_idx + list(wrong_idx[:2])
sample_classifications = []
for idx in sample_indices:
    sample_classifications.append({
        "text": test_df.iloc[idx]["text"],
        "true_label": ID_TO_LABEL[ft_true_ids[idx]],
        "predicted_label": ID_TO_LABEL[ft_pred_ids[idx]],
        "confidence": float(ft_probs[idx][ft_pred_ids[idx]]),
        "correct": bool(ft_pred_ids[idx] == ft_true_ids[idx]),
    })

# Full wrong-predictions list for error analysis (all errors, not just top 15)
wrong_predictions = []
for idx in wrong_idx:
    wrong_predictions.append({
        "text": test_df.iloc[idx]["text"],
        "true_label": ID_TO_LABEL[ft_true_ids[idx]],
        "predicted_label": ID_TO_LABEL[ft_pred_ids[idx]],
        "confidence": float(ft_probs[idx][ft_pred_ids[idx]]),
    })

results = {
    "model_info": {
        "base_model": MODEL_NAME,
        "num_labels": NUM_LABELS,
        "labels": list(LABEL_MAP.keys()),
        "train_size": len(train_df),
        "val_size": len(val_df),
        "test_size": len(test_df),
        "hyperparameters": {
            "num_train_epochs": training_args.num_train_epochs,
            "learning_rate": training_args.learning_rate,
            "per_device_train_batch_size": training_args.per_device_train_batch_size,
        },
    },
    "fine_tuned": {
        "accuracy": float(ft_accuracy),
        "per_class_report": ft_report,
    },
    "baseline": {
        "model": "llama-3.3-70b-versatile",
        "accuracy": float(bl_accuracy),
        "per_class_report": bl_report,
        "unparseable_count": none_count,
        "evaluated_on": len(valid),
    },
    "confusion_matrix": cm_dict,
    "sample_classifications": sample_classifications,
    "wrong_predictions": wrong_predictions,
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("✅ Saved: evaluation_results.json")
print(f"   Fine-tuned accuracy:  {ft_accuracy:.3f}")
print(f"   Baseline accuracy:    {bl_accuracy:.3f}")
print(f"   Wrong predictions:    {len(wrong_predictions)} / {len(ft_true_ids)}")
print(f"   Sample entries saved: {len(sample_classifications)}")
print("\nDownload both files via the Files panel (📁) → right-click → Download")
print("Then commit them to your GitHub repo.")

from google.colab import files as colab_files
colab_files.download("evaluation_results.json")
colab_files.download("confusion_matrix.png")